# 05 — Final Evaluation & Model Comparison

This notebook:
1. Compares all trained models side-by-side
2. Shows per-emotion performance breakdown
3. Applies **SHAP** to explain CNN predictions
4. Error analysis: which emotions are confused most

**Prerequisite:** run notebooks 02, 03, 04 (or `run_all.bat`) first.

In [ ]:
import sys; sys.path.insert(0, '..')

import json, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml
from addict import Dict
from sklearn.metrics import confusion_matrix

from data_classes.ravdess_dataset import RAVDESSDataset
from model_classes.cnn_model import CNNEmotionClassifier
from model_classes.rnn_model import SequenceEmotionClassifier
from model_classes.baseline_model import MLPEmotionClassifier
from utils import build_model, dataset_mode, set_seed, EMOTION_NAMES

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

with open('../config/default.yaml') as f:
    cfg = Dict(yaml.safe_load(f))

set_seed(cfg.training.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Metrics from All Experiments

In [ ]:
model_names = ['baseline', 'mlp', 'cnn', 'lstm', 'gru']
rows = []
for name in model_names:
    path = f'../results/metrics_{name}.json'
    if os.path.exists(path):
        m = json.load(open(path))
        rows.append({'Model': name.upper(), **m})

results_df = pd.DataFrame(rows).set_index('Model')
results_df.columns = ['Accuracy', 'F1 Macro', 'F1 Weighted']
display(results_df.round(4).style.highlight_max(color='#d4edda'))

## 2. Bar Chart Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_cols = ['Accuracy', 'F1 Macro', 'F1 Weighted']
colors = sns.color_palette('Set2', len(results_df))

for ax, col in zip(axes, metrics_cols):
    bars = ax.bar(results_df.index, results_df[col], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, results_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.axhline(0.8, color='red', linestyle='--', alpha=0.4, label='0.8 target')

plt.suptitle('Model Comparison on Test Set', fontsize=13)
plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150)
plt.show()

## 3. Side-by-Side Confusion Matrices

In [ ]:
def load_and_predict(model_name):
    ckpt = f'../saved_models/best_{model_name}.pth'
    if not os.path.exists(ckpt): return None, None

    cfg.model.type = model_name
    model = build_model(cfg).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    from torch.utils.data import DataLoader
    mode = dataset_mode(model_name)
    test_ds = RAVDESSDataset(
        cfg.data.data_dir, actor_ids=list(cfg.data.test_actors), mode=mode,
        sample_rate=cfg.data.sample_rate, duration=cfg.data.duration,
        n_mfcc=cfg.data.n_mfcc, n_mels=cfg.data.n_mels,
        n_fft=cfg.data.n_fft, hop_length=cfg.data.hop_length,
    )
    loader = torch.utils.data.DataLoader(test_ds, batch_size=32, num_workers=0)
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return np.array(preds), np.array(labels)

deep_models = ['cnn', 'lstm', 'gru', 'mlp']
predictions = {}
for m in deep_models:
    p, l = load_and_predict(m)
    if p is not None:
        predictions[m] = (p, l)
        print(f'{m.upper()}: loaded')

In [ ]:
n = len(predictions)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for ax, (name, (preds, labels)) in zip(axes, predictions.items()):
    cm = confusion_matrix(labels, preds)
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=EMOTION_NAMES, yticklabels=EMOTION_NAMES,
                ax=ax, cbar=False)
    ax.set_title(name.upper())
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Normalized Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('../results/confusion_matrices_all.png', dpi=150)
plt.show()

## 4. Per-Emotion F1 Breakdown

In [ ]:
from sklearn.metrics import f1_score

per_emotion = {}
for name, (preds, labels) in predictions.items():
    per_emotion[name] = f1_score(labels, preds, average=None, zero_division=0)

emo_df = pd.DataFrame(per_emotion, index=EMOTION_NAMES).T

fig, ax = plt.subplots(figsize=(11, 5))
emo_df.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', 6), edgecolor='black', linewidth=0.4)
ax.set_ylim(0, 1.1)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Emotion F1 Score by Model')
ax.legend(title='Emotion', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../results/per_emotion_f1.png', dpi=150)
plt.show()
display(emo_df.round(3))

## 5. SHAP Explanation (CNN)

We use `shap.GradientExplainer` to identify which regions of the mel-spectrogram
most influence CNN predictions for each emotion class.

In [ ]:
import shap

cfg.model.type = 'cnn'
cnn_model = build_model(cfg).to(device)
cnn_model.load_state_dict(torch.load('../saved_models/best_cnn.pth', map_location=device))
cnn_model.eval()

test_ds = RAVDESSDataset(
    cfg.data.data_dir, actor_ids=list(cfg.data.test_actors), mode='melspec',
    sample_rate=cfg.data.sample_rate, duration=cfg.data.duration,
    n_mfcc=cfg.data.n_mfcc, n_mels=cfg.data.n_mels,
    n_fft=cfg.data.n_fft, hop_length=cfg.data.hop_length,
)

# Background (50 random samples) + test samples (one per emotion)
bg_idxs = np.random.choice(len(test_ds), 50, replace=False)
bg_data = torch.stack([test_ds[i][0] for i in bg_idxs]).to(device)

# One sample per class
class_samples = {}
for i in range(len(test_ds)):
    x, label = test_ds[i]
    if label not in class_samples:
        class_samples[label] = x
    if len(class_samples) == cfg.data.n_classes:
        break

test_data = torch.stack([class_samples[k] for k in sorted(class_samples.keys())]).to(device)

explainer = shap.GradientExplainer(cnn_model, bg_data)
shap_values = explainer.shap_values(test_data)   # list of [n, 1, H, W] per class
print('SHAP values computed. Shape per class:', np.array(shap_values[0]).shape)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, emo in enumerate(EMOTION_NAMES):
    shap_img = np.array(shap_values[i])[i, 0]    # SHAP for sample i, class i, channel 0
    orig_img  = test_data[i, 0].cpu().numpy()

    axes[i].imshow(orig_img, aspect='auto', origin='lower', cmap='magma', alpha=0.7)
    im = axes[i].imshow(shap_img, aspect='auto', origin='lower',
                         cmap='RdBu_r', alpha=0.6,
                         vmin=-np.abs(shap_img).max(), vmax=np.abs(shap_img).max())
    axes[i].set_title(f'{emo.capitalize()}')
    axes[i].set_xlabel('Time frame')
    axes[i].set_ylabel('Mel band')
    fig.colorbar(im, ax=axes[i], fraction=0.03)

plt.suptitle('SHAP Explanations — CNN on Mel-Spectrograms\n(red = increases predicted class, blue = decreases)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/shap_cnn.png', dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
print('=== FINAL RESULTS SUMMARY ===')
print(results_df.round(4).to_string())

best_model = results_df['F1 Macro'].idxmax()
print(f'\nBest model by F1 Macro: {best_model} ({results_df.loc[best_model,"F1 Macro"]:.4f})')